# Data Cleaning & Preprocessing: Career Path Decision Tree Training Data

In [116]:
import pandas as pd

df = pd.read_csv('results.csv', low_memory=False)
print(f"Starting shape: {df.shape[0]} rows, {df.shape[1]} columns")

Starting shape: 49191 rows, 172 columns


Of the 172 raw columns in the survey export, most describe a **working developer's current job situation** -- salary, employer, remote work setup, tool opinions, and similar. A student building a roadmap has none of that information about themselves yet, so those columns are useless as prediction inputs. Only 8 columns describe things a student *can* actually provide about their own background and skills, so those are the only ones kept.

In [117]:
keep_cols = [
    'MainBranch', 'EdLevel', 'YearsCode', 'LanguageHaveWorkedWith',
    'DatabaseHaveWorkedWith', 'PlatformHaveWorkedWith',
    'WebframeHaveWorkedWith', 'DevType',
]

dropped_count = df.shape[1] - len(keep_cols)
df = df[keep_cols]
print(f"Columns kept: {len(keep_cols)}, columns dropped: {dropped_count}")

Columns kept: 8, columns dropped: 164


Only respondents who answered `MainBranch == "I am a developer by profession"` give a real skill-to-role mapping -- they actually work in the role their skills led them to. Students, hobbyists, and retired respondents would add noise rather than signal, since their DevType/skill relationship isn't the kind of career outcome this tool wants to predict.

In [118]:
before = len(df)
df = df[df['MainBranch'] == 'I am a developer by profession']
after = len(df)
print(f"Rows before: {before}, rows after: {after}")

Rows before: 49191, rows after: 37467


Five `DevType` categories aren't valid targets for a roadmap tool to recommend a student toward:
- **Student** -- circular, it's not a career outcome
- **Retired** -- not a role to grow into
- **Senior executive (C-suite, VP, etc.)** -- reached via tenure, not a skill profile
- **Founder, technology or otherwise** -- same issue, and too vague as a category
- **Other (please specify):** -- undefined, no consistent skill signal

These rows are dropped.

In [119]:
invalid_devtypes = [
    'Student',
    'Retired',
    'Senior executive (C-suite, VP, etc.)',
    'Founder, technology or otherwise',
    'Other (please specify):',
]

before = len(df)
removed_mask = df['DevType'].isin(invalid_devtypes)
removed_present = sorted(df.loc[removed_mask, 'DevType'].unique().tolist())
df = df[~removed_mask]
after = len(df)
print(f"Rows before: {before}, rows after: {after}")
print(f"Categories removed: {removed_present}")

Rows before: 37467, rows after: 35546
Categories removed: ['Founder, technology or otherwise', 'Other (please specify):', 'Retired', 'Senior executive (C-suite, VP, etc.)', 'Student']


`EdLevel`'s "Other (please specify):" response can't be placed on an ordinal education scale -- there's no way to know where it falls relative to the other levels. Those rows are dropped too.

In [120]:
before = len(df)
df = df[df['EdLevel'] != 'Other (please specify):']
after = len(df)
print(f"Rows before: {before}, rows after: {after}")

Rows before: 35546, rows after: 35200


This step takes a **complete-case** approach: any row missing a value in one of the 6 feature columns or `DevType` is dropped, rather than imputing a value. The dataset is large enough that this doesn't meaningfully hurt the sample size, and imputing categorical skill/education data would risk inventing signal that isn't really there.

In [121]:
required_cols = [
    'EdLevel', 'YearsCode', 'LanguageHaveWorkedWith', 'DatabaseHaveWorkedWith',
    'PlatformHaveWorkedWith', 'WebframeHaveWorkedWith', 'DevType',
]

before = len(df)
df = df.dropna(subset=required_cols)
after = len(df)
print(f"Rows before: {before}, rows after: {after}")

Rows before: 35200, rows after: 14208


`DevType` classes with too few examples can't be reliably learned or evaluated on -- a decision tree (and any train/test split) needs enough examples per class to find a pattern and to check it. A **50-example minimum** keeps every major technical career path while dropping classes too small to mean anything statistically.

In [122]:
before = len(df)
class_counts = df['DevType'].value_counts()
small_classes = class_counts[class_counts < 50].index.tolist()

df = df[~df['DevType'].isin(small_classes)]
after = len(df)

print(f"Rows before: {before}, rows after: {after}")
print(f"Final class count: {df['DevType'].nunique()}")
print(f"Classes dropped (< 50 rows): {small_classes}")
print("\nRemaining class distribution:")
print(df['DevType'].value_counts())

Rows before: 14208, rows after: 14018
Final class count: 18
Classes dropped (< 50 rows): ['Project manager', 'Applied scientist', 'System administrator', 'Support engineer or analyst', 'Database administrator or engineer', 'Product manager', 'Data or business analyst', 'Financial analyst or engineer', 'UX, Research Ops or UI design professional']

Remaining class distribution:
DevType
Developer, full-stack                            6665
Developer, back-end                              2969
Architect, software or solutions                 1191
Developer, front-end                              546
Developer, desktop or enterprise applications     492
Engineering manager                               413
DevOps engineer or professional                   350
Data engineer                                     250
Developer, mobile                                 227
AI/ML engineer                                    206
Developer, embedded applications or devices       128
Cloud infrastructu

`YearsCode` has implausible outliers -- some respondents entered 100, and this shows up across every age bracket including 18-24 year olds, which clearly isn't a real answer. Rather than build a complex per-respondent validation against age, this caps `YearsCode` at **40 years** as a simple, defensible senior-career ceiling.

In [123]:
print("YearsCode summary before capping:")
print(df['YearsCode'].describe())

df['YearsCode'] = df['YearsCode'].clip(upper=40)

print("\nYearsCode summary after capping:")
print(df['YearsCode'].describe())

YearsCode summary before capping:
count    14018.000000
mean        17.588101
std         10.285799
min          1.000000
25%         10.000000
50%         15.000000
75%         25.000000
max        100.000000
Name: YearsCode, dtype: float64

YearsCode summary after capping:
count    14018.000000
mean        17.447425
std          9.909823
min          1.000000
25%         10.000000
50%         15.000000
75%         25.000000
max         40.000000
Name: YearsCode, dtype: float64


`EdLevel` gets an **ordinal encoding**, specifically -- unlike the skill columns, it has a real order (doctorate > master's > bachelor's > ...). Mapping it to a single numeric column (1-7) preserves that order and lets the decision tree ask one meaningful threshold question (e.g. "EdLevel >= 5?"), rather than one-hot exploding it into 7 unordered columns that hide the ranking.

In [124]:
edlevel_map = {
    'Primary/elementary school': 1,
    'Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)': 2,
    'Some college/university study without earning a degree': 3,
    'Associate degree (A.A., A.S., etc.)': 4,
    'Bachelor\u2019s degree (B.A., B.S., B.Eng., etc.)': 5,
    'Master\u2019s degree (M.A., M.S., M.Eng., MBA, etc.)': 6,
    'Professional degree (JD, MD, Ph.D, Ed.D, etc.)': 7,
}

sample_before = df[['EdLevel']].sample(5, random_state=42).copy()
df['EdLevel'] = df['EdLevel'].map(edlevel_map)
sample_after = df.loc[sample_before.index, 'EdLevel']

comparison = sample_before.copy()
comparison['EdLevel_encoded'] = sample_after
print(comparison)
print(f"\nUnmapped (NaN) values after mapping: {df['EdLevel'].isna().sum()}")

                                               EdLevel  EdLevel_encoded
16669     Bachelor’s degree (B.A., B.S., B.Eng., etc.)                5
2363   Master’s degree (M.A., M.S., M.Eng., MBA, etc.)                6
37316     Bachelor’s degree (B.A., B.S., B.Eng., etc.)                5
14212     Bachelor’s degree (B.A., B.S., B.Eng., etc.)                5
12228     Bachelor’s degree (B.A., B.S., B.Eng., etc.)                5

Unmapped (NaN) values after mapping: 0


The 4 skill columns (`LanguageHaveWorkedWith`, `DatabaseHaveWorkedWith`, `PlatformHaveWorkedWith`, `WebframeHaveWorkedWith`) get **multi-hot encoding**. These are semicolon-joined multi-select checkbox answers -- not ordered, and not mutually exclusive (a respondent can know both Python and Java). Each possible skill becomes its own binary column via `str.get_dummies(sep=';')`, prefixed with the original column name so the source is traceable.

In [125]:
skill_cols = [
    'LanguageHaveWorkedWith', 'DatabaseHaveWorkedWith',
    'PlatformHaveWorkedWith', 'WebframeHaveWorkedWith',
]

for col in skill_cols:
    dummies = df[col].str.get_dummies(sep=';')
    dummies = dummies.add_prefix(f"{col}__")
    df = pd.concat([df, dummies], axis=1)
    df = df.drop(columns=[col])

print(f"Final shape after multi-hot encoding: {df.shape}")

Final shape after multi-hot encoding: (14018, 146)


The dataset is now fully numeric and ready for a CART decision tree. No scaling is needed -- tree splits are threshold-based ("is this value <= X?") and scale-invariant, unlike distance-based or gradient-based models.

`MainBranch` was only used earlier as a **filter** (Cell 6, keeping only "I am a developer by profession" rows) -- it was never meant to be a feature. Since that filter left every remaining row with the identical value, the column is now constant and carries zero information for the tree to split on. It's dropped here, right before the final save.

In [126]:
print(f"Columns before drop: {df.shape[1]}")
df = df.drop(columns=['MainBranch'])
print(f"Columns after drop: {df.shape[1]}")

Columns before drop: 146
Columns after drop: 145


In [127]:
print(f"Final shape: {df.shape}")
print(f"\nFinal DevType class count: {df['DevType'].nunique()}")
print("Final DevType distribution:")
print(df['DevType'].value_counts())

total_missing = df.isna().sum().sum()
print(f"\nTotal missing values remaining: {total_missing}")
assert total_missing == 0, "Missing values remain!"

df.to_csv('training_data_final.csv', index=False)
print("\nSaved to training_data_final.csv")

Final shape: (14018, 145)

Final DevType class count: 18
Final DevType distribution:
DevType
Developer, full-stack                            6665
Developer, back-end                              2969
Architect, software or solutions                 1191
Developer, front-end                              546
Developer, desktop or enterprise applications     492
Engineering manager                               413
DevOps engineer or professional                   350
Data engineer                                     250
Developer, mobile                                 227
AI/ML engineer                                    206
Developer, embedded applications or devices       128
Cloud infrastructure engineer                     128
Data scientist                                    109
Academic researcher                                84
Developer, AI apps or physical AI                  79
Cybersecurity or InfoSec professional              63
Developer, QA or test                      

A **stratified train/test split** must happen before anything else in building the tree -- specifically before computing class weights, since weights need to come from the training set's class counts only. Splitting after weighting would let information about the test set's distribution leak into training.

"Stratified" means each class keeps its proportion in both the train and test sets. This matters here because the smallest class (`Developer, game or graphics`, 55 rows) could otherwise land almost entirely in one set by chance with a plain random split -- a stratified split guarantees roughly 80% of it lands in train and 20% in test, same as every other class.

In [128]:
import pandas as pd
from sklearn.model_selection import train_test_split

data = pd.read_csv('training_data_final.csv', low_memory=False)

X = data.drop(columns=['DevType'])
y = data['DevType']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

proportions = pd.DataFrame({
    'train_pct': y_train.value_counts(normalize=True) * 100,
    'test_pct': y_test.value_counts(normalize=True) * 100,
}).sort_values('train_pct', ascending=False)

print("\nClass proportions, train vs test (%):")
print(proportions.round(2))

X_train shape: (11214, 144)
X_test shape: (2804, 144)

Class proportions, train vs test (%):
                                               train_pct  test_pct
DevType                                                           
Developer, full-stack                              47.55     47.54
Developer, back-end                                21.18     21.18
Architect, software or solutions                    8.50      8.49
Developer, front-end                                3.90      3.89
Developer, desktop or enterprise applications       3.51      3.50
Engineering manager                                 2.94      2.96
DevOps engineer or professional                     2.50      2.50
Data engineer                                       1.78      1.78
Developer, mobile                                   1.62      1.60
AI/ML engineer                                      1.47      1.46
Developer, embedded applications or devices         0.91      0.93
Cloud infrastructure engineer       

**Class weighting** corrects for the class imbalance visible above -- with `Developer, full-stack` at 6,665 rows and `Developer, game or graphics` at 55, an unweighted tree would learn to favor predicting full-stack whenever uncertain, since that minimizes error on the training data even while ignoring the small classes entirely.

Weighting each class **inversely to its frequency** (computed from `y_train` only, so no test-set information leaks in) means a misclassified example from a rare class costs more than one from a common class -- this is what the Gini/entropy impurity calculation will use in the next step to build the tree.

In [129]:
class_counts_train = y_train.value_counts()
num_classes = len(class_counts_train)

class_weights = {
    cls: len(y_train) / (num_classes * count)
    for cls, count in class_counts_train.items()
}

sorted_weights = dict(sorted(class_weights.items(), key=lambda kv: kv[1], reverse=True))

print("Class weights (highest to lowest):")
for cls, w in sorted_weights.items():
    print(f"  {cls:55s} {w:.4f}")

Class weights (highest to lowest):
  Developer, game or graphics                             14.1591
  Cybersecurity or InfoSec professional                   12.4600
  Developer, QA or test                                   12.2157
  Developer, AI apps or physical AI                       9.8889
  Academic researcher                                     9.2985
  Data scientist                                          7.1609
  Developer, embedded applications or devices             6.1078
  Cloud infrastructure engineer                           6.1078
  AI/ML engineer                                          3.7758
  Developer, mobile                                       3.4231
  Data engineer                                           3.1150
  DevOps engineer or professional                         2.2250
  Engineering manager                                     1.8879
  Developer, desktop or enterprise applications           1.5812
  Developer, front-end                              

**Gini impurity**, in plain terms, is a number from 0 to just under 1 measuring how "mixed up" a group of labels is. 0 means every example in the group is the same class (perfectly pure). Higher means more mixed -- a group evenly split across many classes has impurity close to 1. The tree's whole job at each step is finding a question that splits a mixed group into two less-mixed groups.

Since class weights were computed in the previous step, this uses the **weighted** version: a mistake involving a rare class should count for more than one involving a common class, so impurity is calculated using weighted class proportions, not raw counts.

In [130]:
import numpy as np

def gini_impurity(y, sample_weights):
    total = sample_weights.sum()
    if total == 0:
        return 0.0
    _, inv = np.unique(y, return_inverse=True)
    class_weight_sums = np.bincount(inv, weights=sample_weights)
    proportions = class_weight_sums / total
    return 1.0 - np.sum(proportions ** 2)

sample_weights_train = y_train.map(class_weights).to_numpy()
y_train_arr = y_train.to_numpy()

full_impurity = gini_impurity(y_train_arr, sample_weights_train)
print(f"Gini impurity of full y_train (18 mixed classes): {full_impurity:.4f}")

single_class_mask = y_train_arr == 'Developer, full-stack'
single_class_impurity = gini_impurity(
    y_train_arr[single_class_mask], sample_weights_train[single_class_mask]
)
print(f"Gini impurity of a single-class subset (Developer, full-stack only): {single_class_impurity:.4f}")

Gini impurity of full y_train (18 mixed classes): 0.9444
Gini impurity of a single-class subset (Developer, full-stack only): 0.0000


**Finding the best split** means trying each feature, and for each feature, trying candidate threshold values, asking: "if I split the data into `<= threshold` and `> threshold`, how much less mixed are the two resulting groups compared to before?" The split that reduces impurity the most (weighted by how large each resulting group is) gets chosen.

Binary skill columns only need one threshold (0.5, i.e. has the skill or doesn't), while `YearsCode` and `EdLevel` need multiple candidate thresholds tried, since they're not binary -- the candidates tried are the midpoints between sorted unique values actually present in the current data.

In [131]:
def best_split(X, y, sample_weights, feature_list):
    total_weight = sample_weights.sum()
    base_impurity = gini_impurity(y, sample_weights)
    best_feature, best_threshold, best_reduction = None, None, 0.0

    for j, feat in enumerate(feature_list):
        col = X[:, j]
        uniq = np.unique(col)
        if uniq.size < 2:
            continue
        if uniq.size == 2 and set(uniq.tolist()) == {0, 1}:
            thresholds = np.array([0.5])
        else:
            thresholds = (uniq[:-1] + uniq[1:]) / 2.0

        for t in thresholds:
            left_mask = col <= t
            left_w = sample_weights[left_mask].sum()
            right_w = total_weight - left_w
            if left_w == 0 or right_w == 0:
                continue
            left_impurity = gini_impurity(y[left_mask], sample_weights[left_mask])
            right_impurity = gini_impurity(y[~left_mask], sample_weights[~left_mask])
            weighted_impurity = (left_w * left_impurity + right_w * right_impurity) / total_weight
            reduction = base_impurity - weighted_impurity
            if reduction > best_reduction:
                best_reduction = reduction
                best_feature = feat
                best_threshold = t

    return best_feature, best_threshold, best_reduction

feature_list = list(X_train.columns)
X_train_arr = X_train[feature_list].to_numpy(dtype=float)

root_feature, root_threshold, root_reduction = best_split(
    X_train_arr, y_train_arr, sample_weights_train, feature_list
)
print(f"Root split: {root_feature} <= {root_threshold}, impurity reduction = {root_reduction:.4f}")

Root split: LanguageHaveWorkedWith__Swift <= 0.5, impurity reduction = 0.0119


**Recursive tree building**: after finding the best split for a group, do the same thing again separately on each of the two resulting groups, and keep going. Three stopping conditions end the recursion, each existing to prevent an overfit tree that just memorizes the training data:

- **`max_depth`** -- don't go deeper than N levels
- **`min_samples_split`** -- don't split a group with fewer than N examples left
- **purity** -- if a group is already 0 impurity, there's nothing left to split

When recursion stops, that becomes a **leaf**, storing the weighted class distribution of whatever examples ended up there.

In [132]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, class_distribution=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.class_distribution = class_distribution

    def is_leaf(self):
        return self.class_distribution is not None


def leaf_distribution(y, sample_weights):
    total = sample_weights.sum()
    dist = {}
    for cls in np.unique(y):
        dist[cls] = sample_weights[y == cls].sum() / total
    return dist


def build_tree(X, y, sample_weights, feature_list, depth=0, max_depth=12, min_samples_split=10):
    if depth >= max_depth or len(y) < min_samples_split or gini_impurity(y, sample_weights) == 0:
        return Node(class_distribution=leaf_distribution(y, sample_weights))

    feature, threshold, reduction = best_split(X, y, sample_weights, feature_list)
    if feature is None or reduction <= 0:
        return Node(class_distribution=leaf_distribution(y, sample_weights))

    j = feature_list.index(feature)
    left_mask = X[:, j] <= threshold
    right_mask = ~left_mask

    left_child = build_tree(X[left_mask], y[left_mask], sample_weights[left_mask],
                             feature_list, depth + 1, max_depth, min_samples_split)
    right_child = build_tree(X[right_mask], y[right_mask], sample_weights[right_mask],
                              feature_list, depth + 1, max_depth, min_samples_split)

    return Node(feature=feature, threshold=threshold, left=left_child, right=right_child)


tree = build_tree(X_train_arr, y_train_arr, sample_weights_train, feature_list)


def tree_stats(node, depth=0):
    if node.is_leaf():
        return depth, 1
    left_depth, left_leaves = tree_stats(node.left, depth + 1)
    right_depth, right_leaves = tree_stats(node.right, depth + 1)
    return max(left_depth, right_depth), left_leaves + right_leaves


max_depth_reached, num_leaves = tree_stats(tree)
print(f"Max depth reached: {max_depth_reached}")
print(f"Total leaf nodes: {num_leaves}")

Max depth reached: 12
Total leaf nodes: 554


A leaf's **class distribution** is not a single predicted label -- it's the weighted proportion of each of the 18 classes among the training examples that ended up at that leaf. This is exactly what produces a top-5 ranked list rather than one guess: sort that distribution and take the top 5.

In [133]:
def predict_proba(tree, sample, feature_list):
    node = tree
    while not node.is_leaf():
        j = feature_list.index(node.feature)
        value = sample[j]
        node = node.left if value <= node.threshold else node.right
    return node.class_distribution


X_test_arr = X_test[feature_list].to_numpy(dtype=float)
sample = X_test_arr[0]
true_label = y_test.iloc[0]

proba = predict_proba(tree, sample, feature_list)
top5 = sorted(proba.items(), key=lambda kv: kv[1], reverse=True)[:5]

print(f"True label: {true_label}\n")
print("Top 5 predicted classes:")
for cls, p in top5:
    marker = "  <-- true label" if cls == true_label else ""
    print(f"  {cls:55s} {p:.4f}{marker}")

True label: Developer, embedded applications or devices

Top 5 predicted classes:
  DevOps engineer or professional                         0.5200
  Developer, back-end                                     0.2452
  Architect, software or solutions                        0.1528
  Developer, full-stack                                   0.0819
